In [1]:
# Load case-study data
import pandas as pd
from pathlib import Path

df = pd.read_parquet("../data/processed/case_study_reviews.parquet")

sample_sizes = {
    "Negative": 4,
    "Neutral/Mixed": 10,
    "Positive": 6
}

c:\Users\Simrat Khalsa\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [ ]:
# Verify the each product has enough observations
availability = (
    df.groupby(["product_id", "rating_group"])
    .size()
    .unstack(fill_value=0)
)

display(availability)

rating_group,Negative,Neutral/Mixed,Positive
product_id,,,
P427421,1199,602,4244
P454806,15,49,590
P467945,15,51,612
P472028,36,106,950
P480162,4,33,574


In [ ]:
# Stratified random sampling
sample_sizes = {
    "Negative" : 4,
    "Neutral/Mixed": 10,
    "Positive": 6   
}

In [7]:
samples = []

for product_id, product_df in df.groupby("product_id"):
    for rating_group, n in sample_sizes.items():
        group = product_df[
            product_df["rating_group"] == rating_group
        ]

        if len(group) < n:
            raise ValueError(
                f"Not enough {rating_group} reviews for {product_id}"
            )

        selected = group.sample(n=n, random_state=42)
        samples.append(selected)

evaluation_df = pd.concat(samples, ignore_index=True)

In [8]:
# Validate the sample
print("Total reviews:", len(evaluation_df))

display(
    evaluation_df.groupby(
        ["product_name", "rating_group"]
    ).size().unstack(fill_value=0)
)

print("Duplicate sampled rows:", evaluation_df.duplicated().sum())

Total reviews: 100


rating_group,Negative,Neutral/Mixed,Positive
product_name,,,
Capture Totale Firming & Wrinkle-Correcting Cream,4,10,6
Capture Totale Super Potent Eye Serum,4,10,6
Capture Totale Super Potent Rich Cream,4,10,6
Cica Recover & Repair Multi-Use Balm,4,10,6
Protini Polypeptide Firming Refillable Moisturizer,4,10,6


Duplicate sampled rows: 0


In [9]:
export_columns = [
    "product_id",
    "product_name",
    "brand_name",
    "rating",
    "rating_group",
    "review_text",
    "review_length_words"
]

evaluation_export = evaluation_df[export_columns].copy()

evaluation_export.insert(
    0,
    "evaluation_id",
    [f"REV_{i:03d}" for i in range(1, len(evaluation_export) + 1)]
)

display(evaluation_export.head())

,evaluation_id,product_id,product_name,brand_name,rating,rating_group,review_text,review_length_words
0,REV_001,P427421,Protini Polypeptide Firming Refillable Moistur...,Drunk Elephant,1,Negative,Normally love Drunk Elephant but this made me ...,53
1,REV_002,P427421,Protini Polypeptide Firming Refillable Moistur...,Drunk Elephant,1,Negative,"I got the mini travel size, and I tried it for...",36
2,REV_003,P427421,Protini Polypeptide Firming Refillable Moistur...,Drunk Elephant,1,Negative,Made me break out very bad!!!! My pores got hu...,12
3,REV_004,P427421,Protini Polypeptide Firming Refillable Moistur...,Drunk Elephant,2,Negative,got a deluxe sample of this in hopes of findin...,32
4,REV_005,P427421,Protini Polypeptide Firming Refillable Moistur...,Drunk Elephant,3,Neutral/Mixed,"Not for dry skin, I’ve heard that this moistur...",43


In [10]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

evaluation_export.to_csv(
    output_dir / "evaluation_sample.csv",
    index=False,
    encoding="utf-8"
)

print("Evaluation sample saved.")

Evaluation sample saved.


In [11]:
display(
    evaluation_export[
        ["product_name", "rating", "review_text"]
    ].sample(10, random_state=7)
)

,product_name,rating,review_text
37,Capture Totale Firming & Wrinkle-Correcting Cream,5,"I’m not sure where to start, the heavenly scen..."
26,Capture Totale Firming & Wrinkle-Correcting Cream,3,I received this as a sample from influenster t...
78,Capture Totale Super Potent Eye Serum,5,i wasn’t expecting to love this as much as i d...
91,Cica Recover & Repair Multi-Use Balm,3,This is a nice product. It’s pretty hydrating ...
49,Capture Totale Super Potent Rich Cream,3,I found the Dior Capture Totale super luxuriou...
15,Protini Polypeptide Firming Refillable Moistur...,5,Protini was the icing on the cake for the DE l...
93,Cica Recover & Repair Multi-Use Balm,3,At first I wasn’t a fan of it because it smell...
71,Capture Totale Super Potent Eye Serum,3,Just ok. Does the same thing cheaper products ...
86,Cica Recover & Repair Multi-Use Balm,3,I received this free to try and review. I have...
22,Capture Totale Firming & Wrinkle-Correcting Cream,2,"Dior is a hit or miss, this was a miss. just s..."


SyntaxError: invalid syntax (2856246870.py, line 1)